# SmartSentry AML — Router

**Single entry point for the whole framework.**

Run all cells. When prompted, type **`yes`** or **`no`**:

| Answer | Runs | Pipeline |
|--------|------|----------|
| **`yes`** | `00__aml_pipeline_orchestrator.ipynb` | Full training — data generation → detector → rules → features → Phase 1 → Phase 2 |
| **`no`**  | `00b__aml_inference_orchestrator.ipynb` | Inference only — rules → features → Phase 1 predict → Phase 2 predict |

The chosen orchestrator runs end-to-end via `nbconvert`. Its live output streams into this notebook, and the fully-executed copy is saved to `outputs_updated/executed_notebooks/` so you can inspect every cell afterwards.

## 1 — Setup

In [2]:
import os
import sys
import time
import subprocess
from datetime import datetime, timedelta

# ── Paths ──────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.getcwd()
OUTPUT_DIR   = os.path.join(os.path.dirname(NOTEBOOK_DIR), "outputs_updated")
EXEC_DIR     = os.path.join(OUTPUT_DIR, "executed_notebooks")
os.makedirs(EXEC_DIR, exist_ok=True)

TRAIN_NOTEBOOK   = os.path.join(NOTEBOOK_DIR, "00__aml_pipeline_orchestrator.ipynb")
PREDICT_NOTEBOOK = os.path.join(NOTEBOOK_DIR, "00b__aml_inference_orchestrator.ipynb")

# Generous timeout — training can be long; inference is quick.
TIMEOUT_MINUTES = 240

print("=" * 70)
print("SmartSentry AML — Router")
print("=" * 70)
print(f"  Notebook directory: {NOTEBOOK_DIR}")
print(f"  Output directory:   {OUTPUT_DIR}")
print(f"  Training target:    {os.path.basename(TRAIN_NOTEBOOK)}",
      "  ✓" if os.path.exists(TRAIN_NOTEBOOK) else "  ⚠ MISSING")
print(f"  Inference target:   {os.path.basename(PREDICT_NOTEBOOK)}",
      " ✓" if os.path.exists(PREDICT_NOTEBOOK) else " ⚠ MISSING")

for _nb in (TRAIN_NOTEBOOK, PREDICT_NOTEBOOK):
    if not os.path.exists(_nb):
        raise FileNotFoundError(
            f"Required orchestrator not found:\n  {_nb}\n\n"
            f"Both 00__aml_pipeline_orchestrator.ipynb and "
            f"00b__aml_inference_orchestrator.ipynb must sit in the same "
            f"directory as this router notebook."
        )
print("\n  Both orchestrators found.")


SmartSentry AML — Router
  Notebook directory: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts
  Output directory:   c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated
  Training target:    00__aml_pipeline_orchestrator.ipynb   ✓
  Inference target:   00b__aml_inference_orchestrator.ipynb  ✓

  Both orchestrators found.


## 2 — Main Function

`run_pipeline(choice)` is the single function that drives everything. It takes `"yes"` or `"no"` (case-insensitive; also accepts `y`/`n`/`train`/`predict`), picks the matching orchestrator, executes it, and streams the output here.

In [3]:
def run_pipeline(choice):
    """Route to the correct orchestrator based on a yes/no answer.

    Parameters
    ----------
    choice : str
        "yes" / "y" / "train"  → run the TRAINING orchestrator (00).
        "no"  / "n" / "predict"→ run the INFERENCE orchestrator (00b).
        Comparison is case-insensitive and whitespace-tolerant.

    Returns
    -------
    dict
        {mode, notebook, exit_code, elapsed_seconds, executed_copy}

    Raises
    ------
    ValueError      if `choice` is not recognised.
    RuntimeError    if the chosen orchestrator exits with a non-zero code.
    """
    answer = str(choice).strip().lower()

    yes_set = {"yes", "y", "true", "1", "train", "t"}
    no_set  = {"no", "n", "false", "0", "predict", "p"}

    if answer in yes_set:
        mode          = "train"
        target_nb     = TRAIN_NOTEBOOK
        description   = "FULL TRAINING pipeline (generation → detector → rules → features → Phase 1 → Phase 2)"
    elif answer in no_set:
        mode          = "predict"
        target_nb     = PREDICT_NOTEBOOK
        description   = "INFERENCE pipeline (rules → features → Phase 1 predict → Phase 2 predict)"
    else:
        raise ValueError(
            f"Unrecognised choice {choice!r}. "
            f"Reply 'yes' (training) or 'no' (inference)."
        )

    print("=" * 70)
    print(f"ROUTER → {mode.upper()} MODE")
    print("=" * 70)
    print(f"  Target notebook: {os.path.basename(target_nb)}")
    print(f"  Pipeline:        {description}")
    print(f"  Started:         {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Timeout:         {TIMEOUT_MINUTES} minutes")
    print()

    executed_copy = os.path.join(EXEC_DIR, os.path.basename(target_nb))

    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--ExecutePreprocessor.timeout=" + str(TIMEOUT_MINUTES * 60),
        "--ExecutePreprocessor.kernel_name=python3",
        "--output", executed_copy,
        target_nb,
    ]

    print("─" * 70)
    print("Live output from the target orchestrator:")
    print("─" * 70)

    start = time.time()

    # Stream stdout/stderr line-by-line so progress is visible here.
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=NOTEBOOK_DIR,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=TIMEOUT_MINUTES * 60 + 120)
    except subprocess.TimeoutExpired:
        proc.kill()
        elapsed = time.time() - start
        print(f"\n⚠ TIMEOUT after {elapsed/60:.1f} min — process killed.")
        raise

    elapsed = time.time() - start

    print("─" * 70)
    print(f"  Finished:      {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Elapsed:       {str(timedelta(seconds=int(elapsed)))}")
    print(f"  Exit code:     {proc.returncode}")
    print(f"  Executed copy: {executed_copy}")
    print("─" * 70)

    if proc.returncode != 0:
        raise RuntimeError(
            f"{mode.upper()} orchestrator exited with code {proc.returncode}.\n"
            f"Open the executed copy to see which cell failed:\n  {executed_copy}"
        )

    print(f"\n✓ {mode.upper()} pipeline completed successfully.")
    return {
        "mode":            mode,
        "notebook":        os.path.basename(target_nb),
        "exit_code":       proc.returncode,
        "elapsed_seconds": round(elapsed, 1),
        "executed_copy":   executed_copy,
    }


print("Main function ready:  run_pipeline(choice)")


Main function ready:  run_pipeline(choice)


## 3 — Choose Mode & Run

The cell below asks for your input and calls `run_pipeline()`. To skip the interactive prompt (e.g. for scheduled/unattended runs), set the environment variable `AML_PIPELINE_MODE` to `train` or `predict` before running.

In [1]:
# Resolve the choice: env var takes precedence, otherwise prompt interactively.
_env_mode = os.environ.get("AML_PIPELINE_MODE", "").strip().lower()

if _env_mode in ("train", "predict"):
    user_choice = "yes" if _env_mode == "train" else "no"
    print(f"AML_PIPELINE_MODE={_env_mode!r} (from environment) → choice = {user_choice!r}")
else:
    try:
        user_choice = input("Run full training pipeline?  (yes / no): ")
    except EOFError:
        # Non-interactive context (e.g. this router itself run via nbconvert
        # with no env var) — default to training.
        user_choice = "no"
        print("Non-interactive context — defaulting to 'no' (training).")

# Single call drives the whole framework.
result = run_pipeline(user_choice)

print()
print("=" * 70)
print("ROUTER COMPLETE")
print("=" * 70)
for k, v in result.items():
    print(f"  {k:<18s}: {v}")


NameError: name 'os' is not defined

## 4 — Where to Look Next

- **`outputs_updated/run_history.jsonl`** — append-only log; the latest run is the last line.
- **`outputs_updated/run_dashboard.csv`** — refreshed monitoring view of every run.
- **Training mode** — model bundles at `python_scripts/ml_outputs/phase1_model_bundle.joblib` and `python_scripts/phase2_outputs/phase2_model_bundle.joblib`.
- **Inference mode** — final predictions at `python_scripts/phase2_outputs/predictions_output.parquet` (+ `.csv`).
- **If a run failed** — open the executed copy under `outputs_updated/executed_notebooks/` and scroll to the cell with the traceback.